In [1]:
import json
from pathlib import Path
import torch
from scgpt.model import TransformerModel
from scgpt.tokenizer import GeneVocab

/software/envs/micromamba/envs/nw-scGPTv2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Definizione dei percorsi
model_dir = Path("/projects/shared/intronic_bam/scGPT_supplements/save/scGPT_human")
model_config_file = model_dir / "args.json"
model_file = model_dir / "best_model.pt"
vocab_file = model_dir / "vocab.json"

In [3]:
# 2. Caricamento del vocabolario dei geni
vocab = GeneVocab.from_file(vocab_file)
special_tokens = ["<pad>", "<cls>", "<eoc>", "<mask>", "<unk>"]
for s in special_tokens:
    if s not in vocab:
        vocab.append_token(s)

# 3. Caricamento della configurazione dell'architettura
with open(model_config_file, "r") as f:
    model_configs = json.load(f)

# 4. Inizializzazione del modello Transformer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
model_configs

{'data_source': '/scratch/ssd004/datasets/cellxgene/scb_strict/human',
 'save_dir': '/scratch/ssd004/datasets/cellxgene/save/cellxgene_census_human-May23-08-36-2023',
 'load_model': None,
 'n_hvg': None,
 'valid_size_or_ratio': 0.003,
 'dist_backend': 'nccl',
 'grad_accu_steps': 1,
 'pad_token': '<pad>',
 'input_style': 'binned',
 'input_emb_style': 'continuous',
 'n_bins': 51,
 'max_seq_len': 1200,
 'training_tasks': 'both',
 'dist_url': 'tcp://gpu188.cluster.local:53833',
 'mask_ratio': [0.25, 0.5, 0.75],
 'trunc_by_sample': True,
 'vocab_path': '/scratch/ssd004/datasets/cellxgene/scFormer/scformer/tokenizer/default_census_vocab.json',
 'rank': 0,
 'batch_size': 32,
 'eval_batch_size': 64,
 'epochs': 6,
 'lr': 0.0001,
 'scheduler_interval': 100,
 'scheduler_factor': 0.99,
 'warmup_ratio_or_step': 10000.0,
 'no_cls': True,
 'no_cce': True,
 'fp16': True,
 'fast_transformer': True,
 'nlayers': 12,
 'nheads': 8,
 'embsize': 512,
 'd_hid': 512,
 'dropout': 0.2,
 'n_layers_cls': 3,
 'log_

In [4]:
model = TransformerModel(
    ntoken=len(vocab),
    d_model=model_configs["embsize"],
    nhead=model_configs["nheads"],
    d_hid=model_configs["d_hid"],
    nlayers=model_configs["nlayers"],
    nlayers_cls=model_configs.get("n_layers_cls", 3),
    vocab=vocab,
    pad_token="<pad>",  # Passa la stringa "<pad>", non l'indice vocab["<pad>"]
    pad_value=model_configs.get("pad_value", 0),
    do_mvc=model_configs.get("GEPC", False),
)


In [19]:
model

TransformerModel(
  (encoder): GeneEncoder(
    (embedding): Embedding(60699, 512, padding_idx=60694)
    (enc_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (value_encoder): ContinuousValueEncoder(
    (dropout): Dropout(p=0.5, inplace=False)
    (linear1): Linear(in_features=1, out_features=512, bias=True)
    (activation): ReLU()
    (linear2): Linear(in_features=512, out_features=512, bias=True)
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=512, bias=True)
        (dropout): Dropout(p=0.5, inplace=False)
        (linear2): Linear(in_features=512, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, el

In [13]:
# 5. Caricamento dei pesi (state dict)
model_dict = model.state_dict()
pretrained_dict = torch.load(model_file, map_location=device)
pretrained_dict = {
    k: v for k, v in pretrained_dict.items() 
    if k in model_dict and v.shape == model_dict[k].shape
}
model_dict.update(pretrained_dict)
model.load_state_dict(model_dict)

model.to(device)
model.eval()

print("Modello caricato con successo su:", device)


Modello caricato con successo su: cuda


In [21]:
batch_size = 5
seq_len = 100

# 1. ID dei geni (interi compresi tra 0 e len(vocab))
src = torch.randint(0, len(vocab), (batch_size, seq_len), dtype=torch.long).to(device)

# 2. Valori di espressione dei geni (float)
values = torch.rand(batch_size, seq_len, dtype=torch.float32).to(device)

# 3. Maschera di padding (tutti False = nessun padding in questo test)
src_key_padding_mask = torch.zeros((batch_size, seq_len), dtype=torch.bool).to(device)

In [22]:
emb_layers = {}
hooks = []

def get_layer_embed(name):
    def hook(module, input, output):
        # Se l'output è una tupla/lista (es. (tensor, attn_weights)), estraiamo il primo elemento
        if isinstance(output, (tuple, list)):
            out_tensor = output[0]
        else:
            out_tensor = output
            
        # Salva solo se è un vero tensore PyTorch
        if isinstance(out_tensor, torch.Tensor):
            emb_layers[name] = out_tensor.detach().cpu()
    return hook

# 1. Registra l'hook SOLO sui blocchi principali dell'encoder
for name, module in model.named_modules():
    if "transformer_encoder.layers." in name and name.split(".")[-1].isdigit():
        hooks.append(module.register_forward_hook(get_layer_embed(name)))

print(f"Hook registrati su {len(hooks)} layer principali.")

# 2. Esegui il forward pass
with torch.no_grad():
    output = model(src.to(device), values.to(device), src_key_padding_mask.to(device))

Hook registrati su 12 layer principali.


In [23]:

# 3. Verifica i layer catturati
for name, tensor in emb_layers.items():
    print(f"Layer: {name}")

# 4. Rimuovi gli hook per liberare la memoria
for h in hooks:
    h.remove()


Layer: transformer_encoder.layers.0
Layer: transformer_encoder.layers.1
Layer: transformer_encoder.layers.2
Layer: transformer_encoder.layers.3
Layer: transformer_encoder.layers.4
Layer: transformer_encoder.layers.5
Layer: transformer_encoder.layers.6
Layer: transformer_encoder.layers.7
Layer: transformer_encoder.layers.8
Layer: transformer_encoder.layers.9
Layer: transformer_encoder.layers.10
Layer: transformer_encoder.layers.11


In [24]:
tensor

nested_tensor([
  tensor([[-0.1180,  0.7334, -1.4082,  ...,  1.0133,  0.0573,  0.2678],
          [-0.1316,  0.7205, -1.3939,  ...,  1.0227,  0.0658,  0.2731],
          [-0.1165,  0.7364, -1.4110,  ...,  1.0056,  0.0685,  0.2593],
          ...,
          [-0.1319,  0.7203, -1.4001,  ...,  1.0212,  0.0789,  0.2689],
          [-0.1193,  0.7188, -1.4124,  ...,  1.0064,  0.0742,  0.2742],
          [-0.1239,  0.7159, -1.4196,  ...,  1.0126,  0.0797,  0.2811]]),
  tensor([[-0.1256,  0.6578, -1.3330,  ...,  0.9677,  0.1114,  0.2742],
          [-0.1136,  0.6626, -1.3404,  ...,  0.9528,  0.1122,  0.2556],
          [-0.1114,  0.6673, -1.3453,  ...,  0.9550,  0.1213,  0.2625],
          ...,
          [-0.1072,  0.6620, -1.3307,  ...,  0.9584,  0.1289,  0.2746],
          [-0.1098,  0.6693, -1.3512,  ...,  0.9484,  0.1062,  0.2796],
          [-0.0948,  0.6743, -1.3679,  ...,  0.9584,  0.1022,  0.2604]]),
  tensor([[-0.1627,  0.7291, -1.4053,  ...,  1.0227,  0.0691,  0.2618],
          [-0.

In [29]:
tensor[4].shape

torch.Size([100, 512])

In [ ]:
for name, tensor in emb_layers.items():
    

In [54]:
from scgpt.data_collator import DataCollator
from torch.utils.data import DataLoader, SequentialSampler
import anndata as an
import numpy as np
import scanpy as sc
import os

In [39]:
adata = an.read_h5ad("/projects/shared/intronic_bam/datasets/anndata/be1.h5ad")

In [41]:
adata

AnnData object with n_obs × n_vars = 29128 × 36753
    obs: 'Sample', 'Barcode', 'sum', 'detected', 'subsets_Mito_sum', 'subsets_Mito_detected', 'subsets_Mito_percent', 'total', 'discard', 'is_train'
    var: 'ID', 'Symbol', 'Type', 'gene_names'
    uns: 'Samples'
    layers: 'counts'

In [46]:
adata.X = adata.layers["counts"].copy()

In [31]:
class Dataset(torch.utils.data.Dataset):
        def __init__(self, count_matrix, gene_ids, batch_ids=None):
            self.count_matrix = count_matrix
            self.gene_ids = gene_ids
            self.batch_ids = batch_ids

        def __len__(self):
            return len(self.count_matrix)

        def __getitem__(self, idx):
            row = self.count_matrix[idx]
            nonzero_idx = np.nonzero(row)[0]
            values = row[nonzero_idx]
            genes = self.gene_ids[nonzero_idx]
            # append <cls> token at the beginning
            genes = np.insert(genes, 0, vocab["<cls>"])
            values = np.insert(values, 0, model_configs["pad_value"])
            genes = torch.from_numpy(genes).long()
            values = torch.from_numpy(values).float()
            output = {
                "id": idx,
                "genes": genes,
                "expressions": values,
            }
            if self.batch_ids is not None:
                output["batch_labels"] = self.batch_ids[idx]
            return output

In [50]:
gene_col = "gene_names"
adata.var["id_in_vocab"] = [
        vocab[gene] if gene in vocab else -1 for gene in adata.var[gene_col]
    ]
gene_ids_in_vocab = np.array(adata.var["id_in_vocab"])

adata = adata[:, adata.var["id_in_vocab"] >= 0]

with open(model_config_file, "r") as f:
        model_configs = json.load(f)

# Binning will be applied after tokenization. A possible way to do is to use the unified way of binning in the data collator.

vocab.set_default_index(vocab["<pad>"])
genes = adata.var[gene_col].tolist()
gene_ids = np.array(vocab(genes), dtype=int)

In [51]:
count_matrix = adata.X
count_matrix = (
    count_matrix if isinstance(count_matrix, np.ndarray) else count_matrix.toarray()
)

# gene vocabulary ids
if gene_ids is None:
    gene_ids = np.array(adata.var["id_in_vocab"])
    assert np.all(gene_ids >= 0)

use_batch_labels = False # cosi fanno loro
if use_batch_labels:
    batch_ids = np.array(adata.obs["batch_id"].tolist())

dataset = Dataset(
            count_matrix, gene_ids, batch_ids if use_batch_labels else None
        )

In [55]:
collator = DataCollator(
            do_padding=True,
            pad_token_id=vocab[model_configs["pad_token"]],
            pad_value=model_configs["pad_value"],
            do_mlm=False,
            do_binning=True,
            max_length=1200,
            sampling=True,
            keep_first_n_tokens=1,
    )
data_loader = DataLoader(
            dataset,
            batch_size=batch_size,
            sampler=SequentialSampler(dataset),
            collate_fn=collator,
            drop_last=False,
            num_workers=min(len(os.sched_getaffinity(0)), batch_size),
            pin_memory=True,
)


In [61]:
batch_prova = next(iter(data_loader))

In [65]:
batch_prova

{'gene': tensor([[60695,  4882,  8766,  ..., 17697, 30370, 33903],
         [60695, 35000, 18150,  ..., 31357, 12713,  8381],
         [60695, 11477,  5450,  ...,  9667,  8351, 16382],
         [60695,  5016,  2888,  ...,  2876, 11398,  7439],
         [60695, 19466,  9674,  ..., 10822,  8421, 33782]]),
 'expr': tensor([[-2., 28.,  2.,  ..., 42., 49.,  3.],
         [-2., 37., 13.,  ...,  4., 25., 38.],
         [-2.,  6., 46.,  ..., 49.,  9., 19.],
         [-2., 12., 28.,  ..., 44., 47., 25.],
         [-2., 14., 35.,  ...,  7., 11., 49.]]),
 'masked_expr': tensor([[-2., 28.,  2.,  ..., 42., 49.,  3.],
         [-2., 37., 13.,  ...,  4., 25., 38.],
         [-2.,  6., 46.,  ..., 49.,  9., 19.],
         [-2., 12., 28.,  ..., 44., 47., 25.],
         [-2., 14., 35.,  ...,  7., 11., 49.]])}

In [63]:
for h in hooks:
    h.remove()

In [64]:
emb_layers = {}
hooks = []

def get_layer_embed(name):
    def hook(module, input, output):
        # Se l'output è una tupla/lista (es. (tensor, attn_weights)), estraiamo il primo elemento
        if isinstance(output, (tuple, list)):
            out_tensor = output[0]
        else:
            out_tensor = output
            
        # Salva solo se è un vero tensore PyTorch
        if isinstance(out_tensor, torch.Tensor):
            emb_layers[name] = out_tensor.detach().cpu()
    return hook

# 1. Registra l'hook SOLO sui blocchi principali dell'encoder
for name, module in model.named_modules():
    if "transformer_encoder.layers." in name and name.split(".")[-1].isdigit():
        hooks.append(module.register_forward_hook(get_layer_embed(name)))
print(f"Hook registrati su {len(hooks)} layer principali.")

Hook registrati su 12 layer principali.


In [66]:
# 2. Esegui il forward pass per un singolo pezzo di dati
input_gene_ids = batch_prova["gene"].to(device)
src_key_padding_mask = input_gene_ids.eq(
    vocab[model_configs["pad_token"]]
)
with torch.no_grad():
    output = model(input_gene_ids, batch_prova["expr"].to(device), src_key_padding_mask)

In [67]:
for name, tensor in emb_layers.items():
    print(f"Layer: {name}")


Layer: transformer_encoder.layers.0
Layer: transformer_encoder.layers.1
Layer: transformer_encoder.layers.2
Layer: transformer_encoder.layers.3
Layer: transformer_encoder.layers.4
Layer: transformer_encoder.layers.5
Layer: transformer_encoder.layers.6
Layer: transformer_encoder.layers.7
Layer: transformer_encoder.layers.8
Layer: transformer_encoder.layers.9
Layer: transformer_encoder.layers.10
Layer: transformer_encoder.layers.11


In [68]:
output

{'mlm_output': tensor([[30.6195, 30.4353, 30.8986,  ..., 30.6305, 30.5628, 30.8683],
         [30.6013, 30.6217, 30.7709,  ..., 30.8691, 30.4700, 30.3813],
         [30.6366, 30.8823, 30.5870,  ..., 30.7734, 30.7179, 30.5848],
         [30.6093, 30.6558, 30.5709,  ..., 30.7507, 30.5996, 30.5080],
         [30.5523, 30.6813, 30.4969,  ..., 30.8389, 30.7754, 30.6548]],
        device='cuda:0'),
 'cell_emb': tensor([[-0.0519,  0.6998, -1.4943,  ...,  0.8077, -0.0271,  0.1091],
         [-0.0620,  0.7171, -1.4510,  ...,  0.8084, -0.0015,  0.1147],
         [-0.0638,  0.7011, -1.4854,  ...,  0.7880, -0.0313,  0.0979],
         [-0.0401,  0.7132, -1.4638,  ...,  0.8073, -0.0124,  0.1123],
         [-0.0522,  0.6905, -1.4685,  ...,  0.8000, -0.0299,  0.1062]],
        device='cuda:0')}

In [ ]:
emb_layers